# Kaggle to Bronze data

In [2]:
import os
import duckdb
import kagglehub

base_project_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion"
bronze_root_path = os.path.join(base_project_path, "data", "bronze")

datasets_to_ingest = [
    {
        "kaggle_handle": "avikumart/analytics-vidhya-nov22-insurance-claims-dataset",
        "target_subfolder": "analytics_vidhya_claims"
    },
    {
        "kaggle_handle": "apoorvasharma03/vehicle-insurance-dataset",
        "target_subfolder": "vehicle_insurance"
    }
]

con = duckdb.connect()

for ds in datasets_to_ingest:
    print(f"\n--- Processing Dataset: {ds['kaggle_handle']} ---")
    dataset_path = kagglehub.dataset_download(ds["kaggle_handle"])
    dataset_bronze_dir = os.path.join(bronze_root_path, ds["target_subfolder"])
    os.makedirs(dataset_bronze_dir, exist_ok=True)
    
    for filename in os.listdir(dataset_path):
        if filename.endswith(".csv"):
            file_full_path = os.path.join(dataset_path, filename).replace("\\", "/")
            table_name = filename.split("_")[0].lower()
            output_file = os.path.join(dataset_bronze_dir, f"{table_name}.parquet").replace("\\", "/")
            
            print(f"Ingesting {filename} -> {output_file}")
            
            con.execute(f"""
                COPY (
                    SELECT *, 
                           CURRENT_TIMESTAMP AS _ingestion_timestamp,
                           '{file_full_path}' AS _source_file
                    FROM read_csv_auto('{file_full_path}')
                ) TO '{output_file}' (FORMAT PARQUET);
            """)

print("\nBronze ingestion with DuckDB complete!")


--- Processing Dataset: avikumart/analytics-vidhya-nov22-insurance-claims-dataset ---
Ingesting sample_submission_KvRh9Sx.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/sample.parquet
Ingesting test_zo1G9sv.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/test.parquet
Ingesting train_qWM28Yl.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/train.parquet

--- Processing Dataset: apoorvasharma03/vehicle-insurance-dataset ---
Ingesting Vehicle_Insurance.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/vehicle_insurance/vehicle.parquet

Bronze ingestion with DuckDB complete!


 # Renaming and Merging files

In [3]:
import pandas as pd

# 1. Define file path
file_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\vehicle_insurance\vehicle.parquet"

# 2. Read the file
df = pd.read_parquet(file_path)

# 3. Define the column mapping
rename_dict = {
    'Gender': 'gender',
    'Age': 'customer_age',
    'Driving_License': 'has_driving_license',
    'Previously_Insured': 'previously_insured',
    'Vehicle_Age': 'vehicle_age',
    'Vehicle_Damage': 'vehicle_damage',
    'Region_Code': 'region',
    'Policy_Sales_Channel': 'sales_channel',
    'Vintage': 'days_since_last_contact',
    'Response': 'accepted_offer'
}

# 4. Rename columns
df.rename(columns=rename_dict, inplace=True)

# 5. Overwrite the original Parquet file
df.to_parquet(file_path, index=False)

print("File successfully updated!")

File successfully updated!


In [4]:
import pandas as pd

# File paths
train_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\train.parquet"
test_path  = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\test.parquet"

# 1. Read both Parquet files
df_train = pd.read_parquet(train_path)
df_test  = pd.read_parquet(test_path)

# 2. Combine (concatenate) them vertically
df_merged = pd.concat([df_train, df_test], ignore_index=True)

# 3. View shape and preview
print(f"Combined Shape: {df_merged.shape}")
print(df_merged.head())

# 4. Save the combined DataFrame to a new Parquet file
output_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data\bronze\analytics_vidhya_claims\combined_claims.parquet"
df_merged.to_parquet(output_path, index=False)

print("Merged file saved successfully!")

Combined Shape: (97655, 46)
  policy_id  policy_tenure  age_of_car  age_of_policyholder area_cluster  \
0   ID00001       0.515874        0.05             0.644231           C1   
1   ID00002       0.672619        0.02             0.375000           C2   
2   ID00003       0.841110        0.02             0.384615           C3   
3   ID00004       0.900277        0.11             0.432692           C4   
4   ID00005       0.596403        0.11             0.634615           C5   

   population_density  make segment model fuel_type  ... is_central_locking  \
0                4990     1       A    M1       CNG  ...              False   
1               27003     1       A    M1       CNG  ...              False   
2                4076     1       A    M1       CNG  ...              False   
3               21622     1      C1    M2    Petrol  ...               True   
4               34738     2       A    M3    Petrol  ...               True   

  is_power_steering is_driver_seat_heigh